In [ ]:
# 🔥 Clean slate
!pip uninstall -y torch torchvision torchaudio fastai -q

# ✅ Stable CUDA + Python 3.10 combo
!pip install -q torch==2.9.0 torchvision==0.24.0 torchaudio==2.9.0 \
  --index-url https://download.pytorch.org/whl/cu126
print("📦 Installing latest Surya-OCR and dependencies...")
!pip install -q --upgrade surya-ocr pdf2image pdfplumber opencv-python-headless pillow pandas openpyxl nltk xlsxwriter
!apt-get install -qq --reinstall poppler-utils
!pip install --upgrade "XlsxWriter>=3.0.3"
print("✅ Installation complete!")
!pip install pdfplumber

In [57]:
!rm -rf /kaggle/working/*

In [58]:
"""
STAGE 1: RAW DATA EXTRACTION
Extracts raw OCR text from PDF and saves to .txt file
"""

import os
import glob
import pdfplumber
import numpy as np
from pdf2image import convert_from_path
from PIL import Image
import time
import threading
from surya.foundation import FoundationPredictor
from surya.detection import DetectionPredictor
from surya.recognition import RecognitionPredictor

# ==========================================
# CONFIG
# ==========================================
INPUT_FOLDER = "/kaggle/input/dataset-voters"
OUTPUT_FOLDER = "/kaggle/working"
DPI = 800
THREADS = 4

print("✅ Stage 1: Raw Data Extraction")

# ==========================================
# GRID DETECTION (Verified Logic)
# ==========================================
def get_boxes_and_header(pdf_path, page_num, img_w, img_h):
    """
    Uses robust length-based filtering to detect voter boxes
    """
    with pdfplumber.open(pdf_path) as pdf:
        page = pdf.pages[page_num]
        lines = page.lines

        def get_filtered_coords(lines_subset, coord_key, len_key, tol=2.0, min_len=50):
            lines_subset.sort(key=lambda x: x[coord_key])
            if not lines_subset: 
                return []

            unique_coords = []
            curr_pos = [lines_subset[0][coord_key]]
            curr_len = [abs(lines_subset[0][len_key[1]] - lines_subset[0][len_key[0]])]

            for l in lines_subset[1:]:
                pos = l[coord_key]
                length = abs(l[len_key[1]] - l[len_key[0]])

                if abs(pos - curr_pos[-1]) <= tol:
                    curr_pos.append(pos)
                    curr_len.append(length)
                else:
                    if max(curr_len) > min_len:
                        unique_coords.append(sum(curr_pos) / len(curr_pos))
                    curr_pos = [pos]
                    curr_len = [length]

            if max(curr_len) > min_len:
                unique_coords.append(sum(curr_pos) / len(curr_pos))

            return unique_coords

        # Get vertical and horizontal lines
        v_raw = [l for l in lines if abs(l["x1"] - l["x0"]) < 1]
        h_raw = [l for l in lines if abs(l["bottom"] - l["top"]) < 1]

        unique_vs = get_filtered_coords(v_raw, "x0", ("top", "bottom"), min_len=50)
        unique_hs = get_filtered_coords(h_raw, "top", ("x0", "x1"), min_len=50)

        if not unique_vs or not unique_hs:
            return [], None

        scale_x = img_w / page.width
        scale_y = img_h / page.height

        # Header region
        header_bottom = int(unique_hs[0] * scale_y)
        header_rect = (0, 0, img_w, max(50, header_bottom - 5))

        # Extract boxes
        boxes = []
        for i in range(len(unique_hs) - 1):
            y_top = unique_hs[i]
            y_bottom = unique_hs[i + 1]
            if y_bottom - y_top < 40:
                continue

            for j in range(len(unique_vs) - 1):
                x_left = unique_vs[j]
                x_right = unique_vs[j + 1]
                if x_right - x_left < 100:
                    continue

                x = int(x_left * scale_x)
                y = int(y_top * scale_y)
                w = int((x_right - x_left) * scale_x)
                h = int((y_bottom - y_top) * scale_y)
                boxes.append((x, y, w, h))

        boxes.sort(key=lambda b: (int(b[1] // 50), b[0]))
        return boxes, header_rect


# ==========================================
# AUTO PAGE DETECTION
# ==========================================
def is_valid_voter_page(first_box_text):
    """Check if page contains voter data by examining first box"""
    import re
    # Look for voter ID pattern
    return bool(re.search(r"[A-Z]{2,}\d+|[A-Z]+\d{3,}|[A-Z]+/[0-9]+/", first_box_text))


# ==========================================
# MAIN EXTRACTION
# ==========================================
print("⏳ Loading Surya OCR Models...")
foundation_predictor = FoundationPredictor()
det_predictor = DetectionPredictor()
rec_predictor = RecognitionPredictor(foundation_predictor=foundation_predictor)
ocr_lock = threading.Lock()
print("✅ Models Loaded!")

pdf_files = glob.glob(os.path.join(INPUT_FOLDER, "*.pdf"))
print(f"📂 Found {len(pdf_files)} PDFs")

for file_idx, pdf_path in enumerate(pdf_files):
    print(f"\n📄 FILE {file_idx + 1}/{len(pdf_files)}: {os.path.basename(pdf_path)}")
    
    base_name = os.path.basename(pdf_path).replace(".pdf", "")
    txt_path = os.path.join(OUTPUT_FOLDER, f"{base_name}_RAW.txt")
    
    try:
        with pdfplumber.open(pdf_path) as p:
            total_pages = len(p.pages)
        
        print(f"ℹ️  Total pages: {total_pages}")
        
        with open(txt_path, "w", encoding="utf-8") as f:
            f.write("--- RAW EXTRACTED VOTER DATA ---\n\n")
            
            processing_started = False
            
            for page_num in range(total_pages):
                try:
                    # Convert page to image
                    img = convert_from_path(
                        pdf_path, 
                        dpi=DPI, 
                        first_page=page_num + 1, 
                        last_page=page_num + 1
                    )[0]
                    
                    boxes, header_rect = get_boxes_and_header(
                        pdf_path, page_num, img.width, img.height
                    )
                    
                    if not boxes:
                        continue
                    
                    # Check first box for voter ID
                    is_valid = False
                    try:
                        b = boxes[0]
                        crop = img.crop((b[0], b[1], b[0] + b[2], b[1] + b[3]))
                        with ocr_lock:
                            preds = rec_predictor([crop], det_predictor=det_predictor)
                        txt_chk = " ".join([l.text for l in preds[0].text_lines])
                        is_valid = is_valid_voter_page(txt_chk)
                    except:
                        pass
                    
                    # Auto-detection logic
                    if not processing_started:
                        if not is_valid:
                            print(f"   ⏭️  Skip Page {page_num+1} (Intro)")
                            continue
                        else:
                            print(f"   ✅ Detection Started at Page {page_num+1}")
                            processing_started = True
                    else:
                        if not is_valid:
                            print(f"   🛑 End Detected at Page {page_num+1}")
                            break
                    
                    print(f"   📄 Processing Page {page_num+1}...")
                    
                    # Extract header
                    header_text = ""
                    if header_rect:
                        hc = img.crop(header_rect)
                        h_preds = rec_predictor([hc], det_predictor=det_predictor)
                        header_text = " ".join([l.text for l in h_preds[0].text_lines])
                    
                    f.write(f"\n=== PAGE {page_num+1} ===\n")
                    f.write(f"HEADER: {header_text}\n\n")
                    
                    # Extract boxes
                    box_count = 0
                    for i, (x, y, w, h) in enumerate(boxes):
                        crop = img.crop((x, y, x + w, y + h))
                        
                        # Skip empty boxes
                        if np.mean(np.array(crop.convert("L"))) > 250:
                            continue
                        
                        try:
                            with ocr_lock:
                                preds = rec_predictor([crop], det_predictor=det_predictor)
                            # Use pipe separator to maintain structure
                            raw_text = " | ".join([l.text for l in preds[0].text_lines])
                            
                            row = (i // 3) + 1
                            col = (i % 3) + 1
                            f.write(f"BOX {row}-{col}: {raw_text}\n")
                            box_count += 1
                        except Exception as e:
                            print(f"      ⚠️  Error box {i}: {e}")
                    
                    print(f"      ✅ Extracted {box_count} boxes")
                    
                except Exception as e:
                    print(f"   ❌ Error on page {page_num+1}: {e}")
                    continue
        
        print(f"💾 Saved: {txt_path}")
        
    except Exception as e:
        print(f"❌ Error processing {pdf_path}: {e}")

print("\n🎉 STAGE 1 COMPLETE!")
print("Next: Run Stage 2 to clean and parse the data")


✅ Stage 1: Raw Data Extraction
⏳ Loading Surya OCR Models...
✅ Models Loaded!
📂 Found 2 PDFs

📄 FILE 1/2: cleaned1.pdf
ℹ️  Total pages: 9



Detecting bboxes: 100%|██████████| 1/1 [00:00<00:00,  5.03it/s]

Recognizing Text: 100%|██████████| 2/2 [00:00<00:00,  5.37it/s]


   ⏭️  Skip Page 1 (Intro)



Detecting bboxes: 100%|██████████| 1/1 [00:00<00:00,  4.99it/s]

Recognizing Text: 100%|██████████| 1/1 [00:00<00:00,  1.74it/s]


   ⏭️  Skip Page 3 (Intro)


...

   📄 Processing Page 7...



Detecting bboxes: 100%|██████████| 1/1 [00:00<00:00,  5.48it/s]

Recognizing Text: 100%|██████████| 4/4 [00:02<00:00,  1.53it/s]

Detecting bboxes: 100%|██████████| 1/1 [00:00<00:00,  5.17it/s]

Recognizing Text: 100%|██████████| 12/12 [00:01<00:00, 11.47it/s]

Detecting bboxes: 100%|██████████| 1/1 [00:00<00:00,  5.09it/s]

Recognizing Text: 100%|██████████| 11/11 [00:01<00:00,  9.42it/s]

Detecting bboxes: 100%|██████████| 1/1 [00:00<00:00,  5.06it/s]

Recognizing Text: 100%|██████████| 12/12 [00:01<00:00, 10.33it/s]

Detecting bboxes: 100%|██████████| 1/1 [00:00<00:00,  4.99it/s]

Recognizing Text: 100%|██████████| 12/12 [00:01<00:00, 10.08it/s]

Detecting bboxes: 100%|██████████| 1/1 [00:00<00:00,  5.13it/s]

Recognizing Text: 100%|██████████| 12/12 [00:01<00:00, 10.70it/s]

Detecting bboxes: 100%|██████████| 1/1 [00:00<00:00,  4.99it/s]

Recognizing Text: 100%|██████████| 11/11 [00:01<00:00,  9.65it/s]

Detecting bboxes: 100%|██████████| 1/1 [00:00<00:00,  4.74it/s]

Recognizing 

      ✅ Extracted 30 boxes



Detecting bboxes: 100%|██████████| 1/1 [00:00<00:00,  2.85it/s]

   🛑 End Detected at Page 8
💾 Saved: /kaggle/working/cleaned2_RAW.txt

🎉 STAGE 1 COMPLETE!
Next: Run Stage 2 to clean and parse the data


In [ ]:
"""
STAGE 2: DATA CLEANING & PARSING
Reads raw .txt files, applies regex cleaning, and parses into structured data
"""

import os
import glob
import re
import pandas as pd

# ==========================================
# CONFIG
# ==========================================
INPUT_FOLDER = "/kaggle/working"  # Where Stage 1 saved _RAW.txt files
OUTPUT_FOLDER = "/kaggle/working"

# OUTPUT FORMAT CONTROL
SAVE_TXT = False   # Save cleaned text files
SAVE_CSV = True    # Save CSV files
SAVE_EXCEL = True  # Save Excel files

print("✅ Stage 2: Data Cleaning & Parsing")

# ==========================================
# CLEANING FUNCTIONS
# ==========================================
def clean_extracted_text(text):
    """
    Apply comprehensive regex fixes to clean OCR text
    """
    if not text:
        return ""
    
    # 1. REMOVE NOISE & TAGS
    # Remove HTML tags, common OCR artifacts, and decorative elements
    text = re.sub(r"<[^>]+>", " ", text)  # HTML tags
    text = re.sub(r"\bPhoto\b|\bAvailable\b", " ", text)  # Photo/Available text
    text = re.sub(r"[·•]", " ", text)  # Bullet points and dots
    text = re.sub(r"\s+", " ", text)  # Normalize whitespace
    
    # 2. FIX KEYWORD TYPOS
    # Gender Typos: लिग, लीग, लंग → लिंग
    text = re.sub(r"(?:लिग|लीग|लंग)\s*[:\.]?", "लिंग :", text)
    
    # Age Typos: विय → वय
    text = re.sub(r"(?:विय)\s*[:\.]?", "वय :", text)
    
    # Address/House No Typos
    text = re.sub(r"(?:घर\s*क्रमाक|घर\s*क्रमांक)\s*[:\.]?", "घर क्रमांक :", text)
    
    # 3. SERIAL NUMBER INJECTION
    def fix_missing_serial(match):
        full_id = match.group(1)
        serial = match.group(2)
        rest = match.group(3)
        if not rest.strip().startswith(serial):
            return f"{full_id} | {serial} | {rest}"
        else:
            return match.group(0)
    
    text = re.sub(r"(\d+/\d+/(\d+))\s*\|\s*(.*)", fix_missing_serial, text)
    
    # 4. GENDER FIX
    def fix_gender(match):
        val = match.group(1).strip()
        return "लिंग : स्त्री" if "स्त्री" in val else "लिंग : पु"
    
    text = re.sub(r"लिंग\s*:\s*([^\s|,\n]*)", fix_gender, text)
    
    # 5. AGE NUMBER FIX
    def fix_age_digits(match):
        prefix = match.group(1)
        digits = match.group(2)
        mapping = {'8': '४', '9': '९', '0': '७', '4': '५', '3': '३', '2': '२'}
        new_digits = "".join([mapping.get(c, c) for c in digits])
        return prefix + new_digits
    
    text = re.sub(r"(वय\s*:\s*)([\d]+)", fix_age_digits, text)
    
    # 6. GLOBAL QUESTION MARK FIX
    text = re.sub(r"[\d२-९]*\?+[\d२-९]*", lambda m: m.group(0).replace("?", "२"), text)
    
    return text


# ==========================================
# PARSING FUNCTIONS
# ==========================================
def parse_header(text):
    """Parse header text into structured data - only extracts fields that exist"""
    data = {}
    
    # --- DIVISION (निवडणूक विभाग) ---
    # Pattern 1: "विभाग : 5" (File 1 format)
    div_match = re.search(r"निवडणूक\s+विभाग\s*:\s*([^निवार्चन]+?)(?=\s*निवार्चन|$)", text)
    if not div_match:
        # Pattern 2: "प्रभाग क्र: 5" (File 2 format)
        div_match = re.search(r"प्रभाग\s+क्र\s*[:\s]+(\d+)", text)
    data['division'] = div_match.group(1).strip() if div_match else ""
    
    # --- ELECTORAL CONSTITUENCY (निवार्चन गण) ---
    # ONLY extract if "निवार्चन गण" explicitly exists
    # Pattern: "निवार्चन गण: 9"
    gan_match = re.search(r"निवार्चन\s+गण\s*:\s*(\d+)", text)
    data['gan'] = gan_match.group(1).strip() if gan_match else ""
    
    # --- PART NUMBER (यादी भाग क्र.) ---
    # Captures entire text from "यादी भाग क्र." until "पत्ता" or end of header
    # Example: "यादी भाग क्र. १४३ : १ - राहुल नगर स्वातंत्र्त्र सैनिक कॉलोनी परभणी शहर"
    part_match = re.search(r"(यादी\s+भाग\s+क्र\..*?)(?=\s*पत्ता|$)", text)
    data['part_no'] = part_match.group(1).strip() if part_match else ""
    
    # --- ADDRESS (पत्ता) ---
    # Only extracts if "पत्ता :" explicitly exists
    # Pattern: "पत्ता : जि.प.प्रा.शाळा घेवंडा"
    addr_match = re.search(r"पत्ता\s*:\s*(.+?)(?=\s*मतदान|$)", text)
    data['address'] = addr_match.group(1).strip() if addr_match else ""
    
    # --- POLLING STATION (मतदान केंद्र) ---
    # Pattern: "मतदान केंद्र : 9 Ghevanda"
    poll_match = re.search(r"मतदान\s+केंद्र\s*:\s*(.+?)(?=\s*$)", text)
    data['polling_station'] = poll_match.group(1).strip() if poll_match else ""
    
    return data


def parse_box_text(text, header_data):
    """Parse box text into structured voter data - handles multiple formats"""
    row_data = {}
    
    # --- ROBUST VOTER ID EXTRACTION ---
    # Pattern 1: Voter ID at start (e.g., "WMJ6725378 95/153/1")
    # Pattern 2: After S-number (e.g., "96/146/47 | NMG6681910")
    vid_match = re.search(r"\b([A-Z]{3}\d{7}|[A-Z]{3}\d{6}|[A-Z]{2}[A-Z0-9]\d{7})\b", text)
    row_data['voter_id'] = vid_match.group(1) if vid_match else ""
    
    # --- ROBUST S-NUMBER EXTRACTION ---
    # Always looks for pattern XX/XXX/XXX
    s_match = re.search(r"(\d{2,3}/\d{2,3}/\d{1,4})", text)
    row_data['s'] = s_match.group(1) if s_match else ""
    
    # --- ROBUST SERIAL NUMBER EXTRACTION ---
    # Method 1: Explicit pipe-separated number (e.g., "| 3 |")
    sr_match = re.search(r"\|\s*(\d{1,4})\s*\|", text)
    if sr_match:
        row_data['sr.no'] = sr_match.group(1)
    else:
        # Method 2: Extract from S-number (last part)
        if row_data['s']:
            parts = row_data['s'].split('/')
            if len(parts) == 3 and parts[2].isdigit():
                row_data['sr.no'] = parts[2]
            else:
                row_data['sr.no'] = ""
        else:
            row_data['sr.no'] = ""
    
    # --- HEADER COLUMNS ---
    row_data['निवार्चन गण'] = header_data.get('gan', '')
    row_data['यादी भाग क्र.'] = header_data.get('part_no', '')
    row_data['पत्ता'] = header_data.get('address', '')
    row_data['मतदान केंद्र'] = header_data.get('polling_station', '')
    
    # --- NAME EXTRACTION ---
    name_match = re.search(r"मतदाराचे पूर्ण[:\s]*([^|]+)", text)
    row_data['मतदाराचे पूर्ण'] = name_match.group(1).strip() if name_match else ""
    
    # --- HOUSE NUMBER EXTRACTION ---
    # Looks for "घर क्रमांक : XXX" or "घर क्रमाक . XXX"
    house_match = re.search(r"घर क्रमां?क\s*[:\.\s]*([^|]+)", text)
    row_data['घर क्रमांक'] = house_match.group(1).strip() if house_match else ""
    
    # --- GENDER EXTRACTION ---
    gender_match = re.search(r"लिंग\s*:\s*([^|]+)", text)
    row_data['लिंग'] = gender_match.group(1).strip() if gender_match else ""
    
    # --- AGE EXTRACTION ---
    age_match = re.search(r"वय\s*[:\s]*([\d०-९]+)", text)
    row_data['वय'] = age_match.group(1).strip() if age_match else ""
    
    # --- RAW HEADER (Full header text for reference) ---
    row_data['header'] = header_data.get('raw_header', '')
    
    # --- DELETED FLAG ---
    row_data['is_deleted'] = "**" if "**" in text else ""
    
    return row_data


def interpolate_serial_numbers(page_results):
    """Fill missing serial numbers by interpolation"""
    valid_indices = []
    for i, row in enumerate(page_results):
        sr = row.get('sr.no', '')
        if sr and sr.isdigit():
            valid_indices.append((i, int(sr)))
    
    if not valid_indices:
        return
    
    for i, row in enumerate(page_results):
        if not row.get('sr.no') or not row['sr.no'].isdigit():
            prev_anchor = next(((idx, v) for idx, v in reversed(valid_indices) if idx < i), None)
            next_anchor = next(((idx, v) for idx, v in valid_indices if idx > i), None)
            
            interpolated_val = None
            if prev_anchor:
                interpolated_val = prev_anchor[1] + (i - prev_anchor[0])
            elif next_anchor:
                interpolated_val = next_anchor[1] - (next_anchor[0] - i)
            
            if interpolated_val and interpolated_val > 0:
                row['sr.no'] = str(interpolated_val)


# ==========================================
# MAIN PROCESSING
# ==========================================
raw_files = glob.glob(os.path.join(INPUT_FOLDER, "*_RAW.txt"))
print(f"📂 Found {len(raw_files)} raw text files")

for file_idx, txt_path in enumerate(raw_files):
    print(f"\n📄 FILE {file_idx + 1}/{len(raw_files)}: {os.path.basename(txt_path)}")
    
    base_name = os.path.basename(txt_path).replace("_RAW.txt", "")
    
    all_results = []
    current_header = {}
    
    try:
        with open(txt_path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                
                if line.startswith("HEADER:"):
                    header_text = line.replace("HEADER:", "").strip()
                    cleaned_header = clean_extracted_text(header_text)
                    current_header = parse_header(cleaned_header)
                    # Store raw header for the 'header' column
                    current_header['raw_header'] = header_text
                
                elif line.startswith("BOX"):
                    box_text = re.sub(r"BOX \d+-\d+:\s*", "", line).strip()
                    cleaned_box = clean_extracted_text(box_text)
                    row = parse_box_text(cleaned_box, current_header)
                    # Only add rows that have a voter ID (filter out headers/noise)
                    if row.get('voter_id', '').strip():
                        all_results.append(row)
        
        if not all_results:
            print("   ⚠️  No data extracted")
            continue
        
        # Interpolate missing serial numbers
        interpolate_serial_numbers(all_results)
        
        # Renumber all rows sequentially
        for i, row in enumerate(all_results, 1):
            row['sr.no'] = str(i)
        
        print(f"   ✅ Parsed {len(all_results)} voters")
        
        # Create DataFrame
        df = pd.DataFrame(all_results)
        
        # --- EXACT COLUMN ORDER (User Requirements) ---
        # 12 columns total
        final_columns = [
            'sr.no',
            's',
            'voter_id',
            'निवार्चन गण',
            'यादी भाग क्र.',
            'पत्ता',
            'मतदान केंद्र',
            'मतदाराचे पूर्ण',
            'घर क्रमांक',
            'लिंग',
            'वय',
            'header',
            'is_deleted'
        ]
        
        # Ensure all columns exist and reorder
        for col in final_columns:
            if col not in df.columns:
                df[col] = ""
        
        df = df[final_columns]
        
        # Save to TXT (cleaned text - optional)
        if SAVE_TXT:
            txt_path = os.path.join(OUTPUT_FOLDER, f"{base_name}_CLEANED.txt")
            with open(txt_path, 'w', encoding='utf-8') as f:
                for result in all_results:
                    f.write(str(result) + '\n')
            print(f"   💾 Saved TXT: {txt_path}")
        
        # Save to CSV
        if SAVE_CSV:
            csv_path = os.path.join(OUTPUT_FOLDER, f"{base_name}_PARSED.csv")
            df.to_csv(csv_path, index=False, encoding='utf-8-sig')
            print(f"   💾 Saved CSV: {csv_path}")
        
        # Save to Excel using manual openpyxl (avoids sheet visibility error)
        if SAVE_EXCEL:
            try:
                from openpyxl import Workbook
                
                xlsx_path = os.path.join(OUTPUT_FOLDER, f"{base_name}_PARSED.xlsx")
                wb = Workbook()
                ws = wb.active
                ws.title = "Voters"
                
                # Write header
                for col_idx, col_name in enumerate(final_columns, 1):
                    ws.cell(row=1, column=col_idx, value=col_name)
                
                # Write data rows
                for row_idx, row_data in enumerate(all_results, 2):
                    for col_idx, col_name in enumerate(final_columns, 1):
                        ws.cell(row=row_idx, column=col_idx, value=row_data.get(col_name, ''))
                
                wb.save(xlsx_path)
                print(f"   💾 Saved Excel: {xlsx_path}")
            except Exception as e:
                print(f"   ⚠️  Excel export failed: {e}")
        
    except Exception as e:
        print(f"   ❌ Error: {e}")

print("\n🎉 STAGE 2 COMPLETE!")
print("All parsed data saved to CSV and Excel files")


✅ Stage 2: Data Cleaning & Parsing
📂 Found 2 raw text files

📄 FILE 1/2: cleaned1_RAW.txt
   ✅ Parsed 90 voters
   💾 Saved CSV: /kaggle/working/cleaned1_PARSED.csv
   💾 Saved Excel: /kaggle/working/cleaned1_PARSED.xlsx

📄 FILE 2/2: cleaned2_RAW.txt
   ✅ Parsed 90 voters
   💾 Saved CSV: /kaggle/working/cleaned2_PARSED.csv
   💾 Saved Excel: /kaggle/working/cleaned2_PARSED.xlsx

🎉 STAGE 2 COMPLETE!
All parsed data saved to CSV and Excel files
